In [ ]:
import json
import time
from pathlib import Path
import requests

# 1. Dizin Yapılandırması (Directory Configuration)
# Doğru klasöre gitmesi için
current_dir = Path.cwd()
if current_dir.name == "notebooks":
    project_root = current_dir.parent
else:
    project_root = current_dir

# Kesin hedef: proje-koku/data/raw
raw_dir = project_root / "data" / "raw"
raw_dir.mkdir(parents=True, exist_ok=True)

print(f"Verilerin kaydedileceği hedef dizin: {raw_dir.resolve()}")

# Dizi Listesi ve API Parametreleri
# 12 ikonik yapımın TVMaze ID eşlemeleri
SHOWS = {
    "The Simpsons": 83,
    "Doctor Who": 210,
    "Friends": 431,
    "Breaking Bad": 169,
    "Game of Thrones": 82,
    "The Office": 526,
    "Brooklyn Nine-Nine": 49,
    "Grey's Anatomy": 67,
    "Supernatural": 71,
    "Stranger Things": 2993,
    "Black Mirror": 305,
    "Sherlock": 335,
}

raw_data = []
print("TVMaze API üzerinden veri toplama süreci başlatıldı...\n")

# Veri Çekme Döngüsü (Data Ingestion)
for name, show_id in SHOWS.items():
    # episodes endpoint'i diziye ait tüm bölüm detaylarını tek seferde döner
    url = f"https://api.tvmaze.com/shows/{show_id}?embed=episodes"
    
    try:
        # Zaman aşımı ve ağ hatası kontrolleri
        response = requests.get(url, timeout=10)
        response.raise_for_status()
        
        show_json = response.json()
        raw_data.append(show_json)
        
        episodes_count = len(show_json.get("_embedded", {}).get("episodes", []))
        print(f"✓ {name:<18} -> {episodes_count} bölüm başarıyla çekildi.")
        
    except requests.exceptions.HTTPError as http_err:
        print(f"✗ HTTP Hatası ({name}): {http_err}")
    except requests.exceptions.RequestException as err:
        print(f"✗ Bağlantı Hatası ({name}): {err}")
        
    # API rate-limit sınırı
    time.sleep(0.3)

# Ham JSON Dosyasının Kaydedilmesi
output_file = raw_dir / "tvmaze_raw.json"
with open(output_file, "w", encoding="utf-8") as f:
    json.dump(raw_data, f, ensure_ascii=False, indent=2)

print("\n" + "=" * 60)
print(f"İşlem tamamlandı! Toplam çekilen dizi: {len(raw_data)}")
print(f"Ham JSON dosyası başarıyla kaydedildi:\n-> {output_file.resolve()}")
print("=" * 60)

Verilerin kaydedileceği hedef dizin: C:\Users\HUAWEI\OneDrive\Masaüstü\API'dan veriye- Marvel Dataseti\data\raw
TVMaze API üzerinden veri toplama süreci başlatıldı...

✓ The Simpsons       -> 808 bölüm başarıyla çekildi.
✓ Doctor Who         -> 153 bölüm başarıyla çekildi.
✓ Friends            -> 236 bölüm başarıyla çekildi.
✓ Breaking Bad       -> 62 bölüm başarıyla çekildi.
✓ Game of Thrones    -> 73 bölüm başarıyla çekildi.
✓ The Office         -> 202 bölüm başarıyla çekildi.
✓ Brooklyn Nine-Nine -> 153 bölüm başarıyla çekildi.
✓ Grey's Anatomy     -> 471 bölüm başarıyla çekildi.
✓ Supernatural       -> 519 bölüm başarıyla çekildi.
✓ Stranger Things    -> 42 bölüm başarıyla çekildi.
✓ Black Mirror       -> 32 bölüm başarıyla çekildi.
✓ Sherlock           -> 12 bölüm başarıyla çekildi.

İşlem tamamlandı! Toplam çekilen dizi: 12
Ham JSON dosyası başarıyla kaydedildi:
-> C:\Users\HUAWEI\OneDrive\Masaüstü\API'dan veriye- Marvel Dataseti\data\raw\tvmaze_raw.json
